# Sub-gate trajectory exploration

Given a segment L(C, G, T) with G = XX(c1)·YY(c2)·ZZ(c3), 
the sub-gate intermediates are **determined** by u (no free variables):

```
C --[ZZ]--> Q1 --[YY]--> Q2 --[XX]--> T
```

where Q1 = ZZ·u·C, Q2 = YY·ZZ·u·C, T = XX·YY·ZZ·u·C = G·u·C.

Questions:
1. What does the induced path C -> Q1 -> Q2 -> T look like in Weyl space?
2. Are any coordinates constrained / fixed along the path?
3. How do degenerate targets (c1≈c2) differ from generic?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from gulps._accelerate import canonical_matrix, weyl_coordinates

def random_su2(rng):
    """Haar-random SU(2)."""
    q = rng.standard_normal(4)
    q /= np.linalg.norm(q)
    a, b, c, d = q
    return np.array([[a+1j*b, c+1j*d], [-c+1j*d, a-1j*b]])

def random_local(rng):
    """Random kron(u1, u0) in SU(2)xSU(2)."""
    return np.kron(random_su2(rng), random_su2(rng))

def canonical_weyl(U):
    """Weyl coordinates, folded into canonical chamber."""
    w = weyl_coordinates(U)
    # weyl_coordinates already canonicalizes
    return w

def subgate_path(C, G_weyl, u):
    """Compute the sub-gate intermediate path for G·u·C.
    
    G = XX(c1)·YY(c2)·ZZ(c3), applied right-to-left:
    C -> ZZ·u·C -> YY·ZZ·u·C -> XX·YY·ZZ·u·C = T
    
    Returns dict with Weyl coords at each stage.
    """
    c1, c2, c3 = G_weyl
    XX = canonical_matrix(c1, 0.0, 0.0)
    YY = canonical_matrix(0.0, c2, 0.0)
    ZZ = canonical_matrix(0.0, 0.0, c3)
    
    uC = u @ C
    after_zz = ZZ @ uC
    after_yy = YY @ after_zz
    after_xx = XX @ after_yy  # = G @ u @ C
    
    return {
        'C': canonical_weyl(C),
        'Q1_zz': canonical_weyl(after_zz),
        'Q2_yy': canonical_weyl(after_yy),
        'T': canonical_weyl(after_xx),
    }

# Weyl chamber vertices for plotting
WEYL_VERTS = {
    'I':    [0, 0, 0],
    'CX':   [0.5, 0, 0],
    'DCX':  [0.25, 0.25, 0],
    'iSWAP':[0.5, 0.5, 0],
    'SWAP': [0.5, 0.25, 0.25],
    'B':    [0.5, 0.25, 0],
}

def draw_weyl_chamber(ax, alpha=0.05):
    """Draw the Weyl chamber tetrahedron."""
    verts = np.array([
        [0,0,0], [0.5,0,0], [0.5,0.5,0], [0.5,0.25,0.25]
    ])
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    faces = [[verts[j] for j in f] for f in [
        [0,1,2], [0,1,3], [0,2,3], [1,2,3]
    ]]
    ax.add_collection3d(Poly3DCollection(faces, alpha=alpha, 
                                          facecolor='lightblue', edgecolor='gray', linewidth=0.5))
    for name, v in WEYL_VERTS.items():
        ax.text(v[0], v[1], v[2], f' {name}', fontsize=7, color='gray')
    ax.set_xlabel('c1'); ax.set_ylabel('c2'); ax.set_zlabel('c3')
    ax.set_xlim(0, 0.55); ax.set_ylim(0, 0.55); ax.set_zlim(0, 0.3)

print("Setup done.")

## Experiment 1: iSWAP sub-gate trajectories

G = iSWAP = XX(0.5)·YY(0.5), ZZ=I. Path: C -[YY]-> Q1 -[XX]-> T.

Sweep random u, fixed generic C. Visualize induced paths in Weyl chamber.

In [ ]:
# --- iSWAP: 2 sub-gates (YY then XX), ZZ=I ---
G_weyl_iswap = (0.5, 0.5, 0.0)
N = 500
rng = np.random.default_rng(0)

# Multiple prefix types
prefixes = {
    'generic (0.3,0.2,0.1)': canonical_matrix(0.3, 0.2, 0.1),
    'CX-face (0.4,0,0)':     canonical_matrix(0.4, 0.0, 0.0),
    'degenerate (0.3,0.3,0)':canonical_matrix(0.3, 0.3, 0.0),
    'identity':               np.eye(4, dtype=complex),
}

fig, axes = plt.subplots(2, 2, subplot_kw={'projection': '3d'}, figsize=(14, 12))
axes = axes.flat

for idx, (label, C) in enumerate(prefixes.items()):
    ax = axes[idx]
    draw_weyl_chamber(ax)
    
    paths = []
    all_Q1 = []
    all_T = []
    for _ in range(N):
        u = random_local(rng)
        p = subgate_path(C, G_weyl_iswap, u)
        paths.append(p)
        all_Q1.append(p['Q1_zz'])  # ZZ=I, so Q1=u@C ... actually for iswap:
        all_T.append(p['T'])
    
    # For iSWAP (c3=0), ZZ=I, so Q1_zz = u@C (just local equiv of C)
    # Q2_yy = YY @ u @ C (the real first intermediate)
    # T = XX @ YY @ u @ C
    
    # Re-collect with proper naming
    all_C = []
    all_M1 = []  # after YY (the first real sub-gate)
    all_T2 = []  # after XX (= T)
    for _ in range(N):
        u = random_local(rng)
        p = subgate_path(C, G_weyl_iswap, u)
        all_C.append(p['C'])
        all_M1.append(p['Q2_yy'])  # after YY(0.5)
        all_T2.append(p['T'])       # after XX(0.5)
    
    all_C = np.array(all_C)
    all_M1 = np.array(all_M1)
    all_T2 = np.array(all_T2)
    
    # Plot C (star), M1 (dots), T (triangles)
    ax.scatter(*all_C[0], color='green', s=100, marker='*', zorder=10, label='C')
    ax.scatter(all_M1[:,0], all_M1[:,1], all_M1[:,2], 
              c='blue', alpha=0.3, s=10, label='M1 (after YY)')
    ax.scatter(all_T2[:,0], all_T2[:,1], all_T2[:,2], 
              c='red', alpha=0.3, s=10, label='T (after XX)')
    
    # Draw some trajectory lines
    for i in range(min(50, N)):
        c = all_C[0]
        m = all_M1[i]
        t = all_T2[i]
        ax.plot([c[0],m[0],t[0]], [c[1],m[1],t[1]], [c[2],m[2],t[2]], 
                'k-', alpha=0.05, linewidth=0.5)
    
    ax.set_title(f'C = {label}')
    ax.legend(fontsize=7, loc='upper left')

plt.suptitle('iSWAP sub-gate paths: C -[YY]-> M1 -[XX]-> T', fontsize=14)
plt.tight_layout()
plt.show()

## Experiment 2: Coordinate statistics

For each sub-gate step, look at the marginal distributions and correlations.
Key question: does any coordinate stay fixed or tightly constrained?

In [ ]:
# --- Coordinate analysis (table only, no histograms) ---
N = 2000

configs = [
    ('iSWAP, generic C',   (0.5, 0.5, 0.0), canonical_matrix(0.3, 0.2, 0.1)),
    ('iSWAP, C=I',         (0.5, 0.5, 0.0), np.eye(4, dtype=complex)),
    ('iSWAP, degen C',     (0.5, 0.5, 0.0), canonical_matrix(0.3, 0.3, 0.0)),
    ('sq2iswap, generic C',(0.25, 0.25, 0.0), canonical_matrix(0.3, 0.2, 0.1)),
    ('B gate, generic C',  (0.5, 0.25, 0.0), canonical_matrix(0.3, 0.2, 0.1)),
    ('CX, generic C',      (0.5, 0.0, 0.0),  canonical_matrix(0.3, 0.2, 0.1)),
    ('generic, generic C', (0.4, 0.3, 0.15), canonical_matrix(0.3, 0.2, 0.1)),
]

print(f"{'Config':<30} | {'M1 std(c1,c2,c3)':<28} | {'T std(c1,c2,c3)':<28} | {'M1 mean(c1,c2,c3)':<28}")
print('-'*120)
for label, G_weyl, C in configs:
    rng = np.random.default_rng(42)
    all_M1, all_T = [], []
    for _ in range(N):
        u = random_local(rng)
        p = subgate_path(C, G_weyl, u)
        all_M1.append(p['Q2_yy'])
        all_T.append(p['T'])
    all_M1 = np.array(all_M1)
    all_T = np.array(all_T)
    m1s = all_M1.std(axis=0)
    ts = all_T.std(axis=0)
    m1m = all_M1.mean(axis=0)
    print(f'{label:<30} | {m1s[0]:.4f} {m1s[1]:.4f} {m1s[2]:.4f}     | {ts[0]:.4f} {ts[1]:.4f} {ts[2]:.4f}     | {m1m[0]:.4f} {m1m[1]:.4f} {m1m[2]:.4f}')

print("\nNote: M1 = YY(c2)·u·C, T = XX(c1)·YY(c2)·u·C = G·u·C")
print("For CX (c2=0), YY=I so M1 = u·C has SAME Weyl as C (u is local).")

## Experiment 3: Pairwise correlations (M1 vs T)

If M1 determines T (or vice versa) through a simple relationship, 
we should see tight correlations in coordinate scatter plots.

In [ ]:
# --- M1 vs T coordinate scatter + M1-T relationship ---
N = 3000
rng = np.random.default_rng(7)

gate_configs = [
    ('iSWAP',     (0.5, 0.5, 0.0)),
    ('sq2iswap',  (0.25, 0.25, 0.0)),
    ('B gate',    (0.5, 0.25, 0.0)),
    ('CX',        (0.5, 0.0, 0.0)),
]
C = canonical_matrix(0.3, 0.2, 0.1)
c_w = canonical_weyl(C)

fig, axes = plt.subplots(len(gate_configs), 4, figsize=(18, 4*len(gate_configs)))

for row, (gname, G_weyl) in enumerate(gate_configs):
    all_M1 = []
    all_T = []
    for _ in range(N):
        u = random_local(rng)
        p = subgate_path(C, G_weyl, u)
        all_M1.append(p['Q2_yy'])
        all_T.append(p['T'])
    all_M1 = np.array(all_M1)
    all_T = np.array(all_T)
    
    # Scatter: M1_ci vs T_ci
    for col in range(3):
        ax = axes[row, col]
        ax.scatter(all_M1[:, col], all_T[:, col], alpha=0.1, s=3, c='steelblue')
        ax.set_xlabel(f'M1 c{col+1}')
        ax.set_ylabel(f'T c{col+1}')
        r = np.corrcoef(all_M1[:, col], all_T[:, col])[0,1]
        ax.set_title(f'{gname}: c{col+1} vs c{col+1}, r={r:.3f}')
        # identity line
        lim = [0, max(all_M1[:, col].max(), all_T[:, col].max()) + 0.02]
        ax.plot(lim, lim, 'r--', alpha=0.3)
    
    # Delta plot: T - M1 (what does the second sub-gate do?)
    ax = axes[row, 3]
    delta = all_T - all_M1
    for i, (c, lbl) in enumerate(zip(['red','green','blue'], ['dc1','dc2','dc3'])):
        ax.hist(delta[:, i], bins=50, alpha=0.5, color=c, label=lbl, density=True)
    ax.set_title(f'{gname}: T - M1 (shift by XX)')
    ax.legend(fontsize=8)
    ax.set_xlabel('delta')

plt.suptitle(f'M1 vs T correlations | C = {c_w}', fontsize=14)
plt.tight_layout()
plt.show()

# Cross-coordinate correlations
print("\nCross-coordinate correlation matrix (M1_ci vs T_cj):")
for gname, G_weyl in gate_configs:
    rng2 = np.random.default_rng(7)
    all_M1 = []
    all_T = []
    for _ in range(N):
        u = random_local(rng2)
        p = subgate_path(C, G_weyl, u)
        all_M1.append(p['Q2_yy'])
        all_T.append(p['T'])
    all_M1 = np.array(all_M1)
    all_T = np.array(all_T)
    
    R = np.zeros((3,3))
    for i in range(3):
        for j in range(3):
            R[i,j] = np.corrcoef(all_M1[:,i], all_T[:,j])[0,1]
    print(f"\n{gname}:")
    print(f"         T_c1    T_c2    T_c3")
    for i in range(3):
        print(f"  M1_c{i+1}  {R[i,0]:+.3f}  {R[i,1]:+.3f}  {R[i,2]:+.3f}")

## Experiment 4: The "fixed variable" question

In the XX polytope paper, for L(C, XX(a), T), one monodromy coordinate is constrained.
Check: for the sub-gate intermediates, is there a functional relationship 
between M1 coords and T coords that reduces the degrees of freedom?

Also check: does M1 always land in a specific polytope / face?

In [ ]:
# --- Check functional relationships between M1 and T ---
from gulps._accelerate import monodromy_from_weyl, weyl_from_monodromy

def w2m(w):
    """Weyl array -> monodromy array."""
    return np.array(monodromy_from_weyl(w[0], w[1], w[2]))

def m2w(m):
    """Monodromy array -> weyl array."""
    return np.array(weyl_from_monodromy(m[0], m[1], m[2]))

N = 5000
rng = np.random.default_rng(123)
C = canonical_matrix(0.3, 0.2, 0.1)
c_w = canonical_weyl(C)
c_m = w2m(c_w)

G_weyl = (0.5, 0.5, 0.0)  # iSWAP

all_M1_w = []
all_T_w = []
all_M1_m = []
all_T_m = []

for _ in range(N):
    u = random_local(rng)
    p = subgate_path(C, G_weyl, u)
    m1 = p['Q2_yy']
    t = p['T']
    all_M1_w.append(m1)
    all_T_w.append(t)
    all_M1_m.append(w2m(m1))
    all_T_m.append(w2m(t))

all_M1_w = np.array(all_M1_w)
all_T_w = np.array(all_T_w)
all_M1_m = np.array(all_M1_m)
all_T_m = np.array(all_T_m)

# Check various linear combinations
print("=== Checking invariants across the sub-gate path ===")
print(f"C_weyl = {c_w}, C_mono = {c_m}\n")

# Sum and difference of coordinates
sum_w = all_M1_w + all_T_w
diff_w = all_T_w - all_M1_w

print("Weyl coordinate sums (M1+T):")
for i in range(3):
    print(f"  c{i+1}: mean={sum_w[:,i].mean():.4f} std={sum_w[:,i].std():.4f}")

print("\nWeyl coordinate diffs (T-M1):")
for i in range(3):
    print(f"  c{i+1}: mean={diff_w[:,i].mean():.4f} std={diff_w[:,i].std():.4f}")

# Monodromy sums/diffs  
sum_m = all_M1_m + all_T_m
diff_m = all_T_m - all_M1_m

print("\nMonodromy coordinate sums (M1+T):")
for i in range(3):
    print(f"  m{i}: mean={sum_m[:,i].mean():.4f} std={sum_m[:,i].std():.4f}")

print("\nMonodromy coordinate diffs (T-M1):")
for i in range(3):
    print(f"  dm{i}: mean={diff_m[:,i].mean():.4f} std={diff_m[:,i].std():.4f}")

# Check: is any linear combination of M1 and T constant?
print("\n=== Searching for linear invariants f(M1,T) = const ===")
from itertools import product as iprod

best_invariants = []
for coeffs in iprod(range(-2, 3), repeat=6):
    a = np.array(coeffs[:3])
    b = np.array(coeffs[3:])
    if np.all(a == 0) and np.all(b == 0):
        continue
    vals = all_M1_w @ a + all_T_w @ b
    if vals.std() < 0.01 and np.abs(vals.mean()) > 0.001:
        best_invariants.append((vals.std(), vals.mean(), coeffs))

best_invariants.sort()
if best_invariants:
    print(f"Found {len(best_invariants)} near-invariants in Weyl coords (std < 0.01):")
    for std, mean, c in best_invariants[:10]:
        a, b = c[:3], c[3:]
        print(f"  {a}*M1_w + {b}*T_w = {mean:.4f} (std={std:.6f})")
else:
    print("No linear invariants found in Weyl coords.")

# Also try in monodromy coords
best_m = []
for coeffs in iprod(range(-2, 3), repeat=6):
    a = np.array(coeffs[:3])
    b = np.array(coeffs[3:])
    if np.all(a == 0) and np.all(b == 0):
        continue
    vals = all_M1_m @ a + all_T_m @ b
    if vals.std() < 0.01 and np.abs(vals.mean()) > 0.001:
        best_m.append((vals.std(), vals.mean(), coeffs))

best_m.sort()
if best_m:
    print(f"\nFound {len(best_m)} near-invariants in monodromy coords:")
    for std, mean, c in best_m[:10]:
        a, b = c[:3], c[3:]
        print(f"  {a}*M1_m + {b}*T_m = {mean:.4f} (std={std:.6f})")
else:
    print("\nNo linear invariants found in monodromy coords.")

# Also include C in the search: a*C + b*M1 + c*T = const?
print("\n=== Searching for f(C,M1,T) = const (including prefix) ===")
best_cmt = []
for coeffs in iprod(range(-2, 3), repeat=9):
    a = np.array(coeffs[:3])
    b = np.array(coeffs[3:6])
    c = np.array(coeffs[6:])
    if np.all(a == 0) and np.all(b == 0) and np.all(c == 0):
        continue
    # C is constant, so a*C is just a shift
    vals = c_w @ a + all_M1_w @ b + all_T_w @ c
    if vals.std() < 0.005 and np.abs(vals.mean()) > 0.001:
        best_cmt.append((vals.std(), vals.mean(), coeffs))

best_cmt.sort()
if best_cmt:
    print(f"Found {len(best_cmt)} near-invariants (std < 0.005):")
    for std, mean, co in best_cmt[:15]:
        a, b, c = co[:3], co[3:6], co[6:]
        print(f"  {a}*C + {b}*M1 + {c}*T = {mean:.4f} (std={std:.6f})")
else:
    print("No linear invariants found involving C, M1, T.")

## Experiment 5: Dimensionality of the (M1, T) image

The map u -> (M1(u), T(u)) goes from 8D (SU(2)x SU(2)) to 6D (two Weyl triples).
But the image might be lower-dimensional. PCA on the joint (M1, T) cloud reveals effective dimension.

In [ ]:
# --- PCA on joint (M1, T) space ---
N = 5000

gate_configs = [
    ('iSWAP',     (0.5, 0.5, 0.0)),
    ('sq2iswap',  (0.25, 0.25, 0.0)),
    ('B gate',    (0.5, 0.25, 0.0)),
    ('CX',        (0.5, 0.0, 0.0)),
    ('generic',   (0.4, 0.3, 0.15)),
]

prefix_configs = [
    ('generic C=(0.3,0.2,0.1)', canonical_matrix(0.3, 0.2, 0.1)),
    ('C=I',                      np.eye(4, dtype=complex)),
    ('degen C=(0.3,0.3,0)',     canonical_matrix(0.3, 0.3, 0.0)),
]

print(f"{'Gate':<15} {'Prefix':<25} | {'PCA singular values (6D joint space)':<50} | {'eff dim'}")
print('='*110)

for pname, C in prefix_configs:
    for gname, G_weyl in gate_configs:
        rng = np.random.default_rng(42)
        
        joint = []  # (M1_c1, M1_c2, M1_c3, T_c1, T_c2, T_c3)
        for _ in range(N):
            u = random_local(rng)
            p = subgate_path(C, G_weyl, u)
            m1 = p['Q2_yy']
            t = p['T']
            joint.append(np.concatenate([m1, t]))
        
        joint = np.array(joint)
        centered = joint - joint.mean(axis=0)
        svs = np.linalg.svd(centered, compute_uv=False)
        svs_norm = svs / svs[0]  # normalize
        
        # Effective dimension: number of SVs > 1% of largest
        eff_dim = np.sum(svs_norm > 0.01)
        
        sv_str = ' '.join(f'{s:.4f}' for s in svs_norm)
        print(f'{gname:<15} {pname:<25} | {sv_str:<50} | {eff_dim}')
    print()

# For the most interesting case, also look at the PCA directions
print("\n=== PCA loadings for iSWAP + generic C ===")
rng = np.random.default_rng(42)
C = canonical_matrix(0.3, 0.2, 0.1)
joint = []
for _ in range(N):
    u = random_local(rng)
    p = subgate_path(C, (0.5, 0.5, 0.0), u)
    joint.append(np.concatenate([p['Q2_yy'], p['T']]))
joint = np.array(joint)
centered = joint - joint.mean(axis=0)
U, S, Vt = np.linalg.svd(centered, full_matrices=False)

labels = ['M1_c1', 'M1_c2', 'M1_c3', 'T_c1', 'T_c2', 'T_c3']
print(f"\n{'PC':<5} {'SV':>8} | {' '.join(f'{l:>8}' for l in labels)}")
print('-'*75)
for i in range(6):
    loadings = ' '.join(f'{v:+.4f}' for v in Vt[i])
    print(f'PC{i+1:<3} {S[i]/S[0]:>8.4f} | {loadings}')

## Prototype: Sub-gate ISA with analytical solve (no GN)

Pipeline:
1. Factor G = XX·YY·ZZ into sub-gate ISA
2. LP finds sub-intermediates (standard monodromy LP on the sub-gate ISA)
3. For each sub-segment L(C_i, XX(θ), M_i): solve analytically via Qiskit XXDecomposer
4. Stitch and verify

The XXDecomposer takes a 4x4 unitary and decomposes into XX-type gates analytically. 
To solve L(prefix, XX(θ), target), form V = CAN(target) · prefix^{-1} and decompose V.

In [ ]:
# --- Step 0: Check what XXDecomposer gives us ---
from qiskit.synthesis import XXDecomposer
from qiskit.quantum_info import Operator
from gulps._accelerate import recover_local_equiv as recover_local_equivalence
from gulps.core.invariants import GateInvariants

# XXDecomposer with a specific basis gate strength
# For XX(0.5) = CX: the interaction strength is pi/4
# basis_fidelity maps strength -> fidelity. strength=pi/2*c1
# Actually let's just check what it does with defaults

# Test: decompose a random unitary with XXDecomposer
rng = np.random.default_rng(42)
from scipy.stats import unitary_group
U_random = unitary_group.rvs(4, random_state=42)
# Make it SU(4)
U_random = U_random / np.linalg.det(U_random)**(1/4)

# Default XXDecomposer uses CX (XX interaction strength pi/4)
decomposer_cx = XXDecomposer(basis_fidelity=1.0)
circ = decomposer_cx(Operator(U_random))
print("Default XXDecomposer circuit:")
print(circ)
print(f"\nReconstructed fidelity: {abs(np.trace(Operator(circ).data.conj().T @ U_random))/4:.15f}")

# Now try with a specific basis gate (e.g., strength for YY(0.5) = pi/4)
# XXDecomposer basis_fidelity: fidelity of the basis gate
# For a perfect gate with strength alpha: fidelity=1.0
# We can pass a dict: {strength: fidelity}
# strength = pi/2 * weyl_c1 for XX-type gates
import math
strength_cx = math.pi / 4  # XX(0.5) has c1=0.5, strength = pi/2 * 0.5 = pi/4
decomposer_half = XXDecomposer(basis_fidelity={strength_cx: 1.0})
circ2 = decomposer_half(Operator(U_random))
print("\nXXDecomposer with explicit CX strength:")
print(circ2)
print(f"Fidelity: {abs(np.trace(Operator(circ2).data.conj().T @ U_random))/4:.15f}")

In [ ]:
# --- End-to-end prototype: analytical sub-gate solve ---
from qiskit.synthesis.two_qubit.two_qubit_decompose import TwoQubitWeylDecomposition
from qiskit.quantum_info import Operator
from gulps._accelerate import recover_local_equiv as recover_local_equivalence

def analytical_subgate_solve(C_matrix, G_weyl, T_weyl, verbose=True):
    """
    Solve L(C, G, T) analytically by peeling sub-gates via KAK.
    
    Approach: form V = CAN(T) · C^{-1}, then peel sub-gates right-to-left.
    Each peel is a KAK decomposition (analytical, O(1)).
    
    Returns: (result_4x4, corrections_list)
    """
    c1, c2, c3 = G_weyl
    
    # Sub-gate ISA (applied right-to-left: ZZ first, then YY, then XX)
    subgates = []
    if abs(c3) > 1e-10:
        subgates.append(('ZZ', c3, canonical_matrix(0.0, 0.0, c3)))
    if abs(c2) > 1e-10:
        subgates.append(('YY', c2, canonical_matrix(0.0, c2, 0.0)))
    if abs(c1) > 1e-10:
        subgates.append(('XX', c1, canonical_matrix(c1, 0.0, 0.0)))
    
    if verbose:
        print(f"G = {' · '.join(f'{n}({v:.3f})' for n, v, _ in reversed(subgates))}")
    
    # V = CAN(T) · C^{-1}: the "gap" to decompose
    CAN_T = canonical_matrix(*T_weyl)
    V = CAN_T @ np.linalg.inv(C_matrix)
    
    if verbose:
        print(f"V = CAN(T)·C^{{-1}}, Weyl(V) = {canonical_weyl(V)}")
    
    # Peel sub-gates from V using KAK
    # V = gate_n · u_n · ... · gate_1 · u_1
    # gate^{-1} · V = u_n · ... = kak_L · CAN(rem) · kak_R
    # So u_n = kak_R, and remaining = kak_L · CAN(rem)
    
    remaining = V.copy()
    corrections = []
    
    for name, ci, gate_mat in reversed(subgates):  # peel from LEFT (outermost first)
        gate_inv = np.linalg.inv(gate_mat)
        product = gate_inv @ remaining
        
        kak = TwoQubitWeylDecomposition(Operator(product))
        remaining_weyl = np.array([kak.a, kak.b, kak.c]) / (np.pi/2)
        
        # product = K1 · CAN(rem) · K2
        # remaining was = gate · product = gate · K1 · CAN(rem) · K2
        # We want: remaining = gate · u · (next_remaining)
        # where u = K1 (left local of product's KAK) ... hmm
        
        # Let me think again. V = XX · u2 · YY · u1 · ZZ · u0
        # Peeling XX from left: XX^{-1} · V = u2 · YY · u1 · ZZ · u0
        # KAK of XX^{-1} · V gives us: K_L · CAN(w) · K_R
        # The local u2 is NOT simply K_L or K_R because the remaining
        # YY · u1 · ZZ · u0 is not a CAN matrix.
        
        # DIFFERENT approach: peel from RIGHT
        # V = XX · u2 · YY · u1 (for iSWAP with 2 sub-gates)  
        # V · u1^{-1} = XX · u2 · YY
        # But we don't know u1...
        
        # The correct approach: KAK gives us the decomposition
        # product = K_L · CAN(a,b,c) · K_R  
        # So remaining = gate · K_L · CAN(a,b,c) · K_R
        # We set u = K_L (absorb into the gate's right side)
        # and next_remaining = CAN(a,b,c) · K_R
        
        # But K_R gets absorbed into the next step's u...
        # Let's just track the full remaining matrix
        
        k_left = np.kron(kak.K1r, kak.K1l)   # left local of product
        k_right = np.kron(kak.K2r, kak.K2l)   # right local of product
        can_rem = canonical_matrix(*remaining_weyl)
        
        # product = k_left · can_rem · k_right
        # remaining_old = gate · product = gate · k_left · can_rem · k_right
        # We define: u_this = k_left, next_remaining = can_rem · k_right
        
        corrections.append({
            'gate': name,
            'gate_matrix': gate_mat,
            'u_local': k_left,
        })
        
        remaining = can_rem @ k_right
        
        if verbose:
            print(f"  Peeled {name}({ci:.3f}): rem Weyl = {remaining_weyl}")
    
    # remaining should be locally equivalent to C
    final_weyl = canonical_weyl(remaining)
    c_weyl = canonical_weyl(C_matrix)
    
    if verbose:
        print(f"\nFinal remaining Weyl: {final_weyl}")
        print(f"C Weyl:              {c_weyl}")
    
    # Reconstruct: product = gate_n · u_n · gate_{n-1} · u_{n-1} · ... · remaining
    # But remaining ≠ C exactly (differs by locals). Need final recovery.
    result = remaining.copy()
    for corr in corrections:
        result = corr['gate_matrix'] @ corr['u_local'] @ result
    
    result_weyl = canonical_weyl(result)
    target_weyl_arr = np.array(T_weyl)
    err = np.max(np.abs(result_weyl - target_weyl_arr))
    
    if verbose:
        print(f"\nResult Weyl:  {result_weyl}")
        print(f"Target Weyl:  {target_weyl_arr}")
        print(f"Linf error:   {err:.2e}")
    
    return result, corrections, err

# --- Test ---
rng = np.random.default_rng(42)
C = canonical_matrix(0.3, 0.2, 0.1)
u = random_local(rng)
G = canonical_matrix(0.5, 0.5, 0.0)
T_matrix = G @ u @ C
T_weyl = tuple(canonical_weyl(T_matrix))

print("=== Test: iSWAP segment ===")
print(f"C_weyl = {canonical_weyl(C)}, T_weyl = {T_weyl}\n")
result, corrections, err = analytical_subgate_solve(C, (0.5, 0.5, 0.0), T_weyl)

In [ ]:
# --- Step 2: Mass test across random targets ---
N_test = 200
rng = np.random.default_rng(0)

gate_configs = [
    ('iSWAP',     (0.5, 0.5, 0.0)),
    ('sq2iswap',  (0.25, 0.25, 0.0)),
    ('B gate',    (0.5, 0.25, 0.0)),
    ('generic',   (0.4, 0.3, 0.15)),
]

prefix_configs = [
    ('generic', canonical_matrix(0.3, 0.2, 0.1)),
    ('C=I',     np.eye(4, dtype=complex)),
]

print(f"{'Gate':<12} {'Prefix':<10} | {'Success':>7} | {'Mean err':>10} | {'Max err':>10} | {'Mean time':>10}")
print('='*75)

import time

for gname, G_weyl in gate_configs:
    for pname, C in prefix_configs:
        successes = 0
        errors = []
        times = []
        
        for i in range(N_test):
            u = random_local(rng)
            G = canonical_matrix(*G_weyl)
            T_matrix = G @ u @ C
            T_w = tuple(canonical_weyl(T_matrix))
            
            t0 = time.perf_counter()
            try:
                result, _ = analytical_subgate_solve(C, G_weyl, T_w, verbose=False)
                dt = time.perf_counter() - t0
                
                result_w = canonical_weyl(result)
                err = np.max(np.abs(np.array(result_w) - np.array(T_w)))
                errors.append(err)
                times.append(dt)
                
                if err < 0.02:
                    successes += 1
            except Exception as e:
                dt = time.perf_counter() - t0
                times.append(dt)
                errors.append(999)
        
        errors = np.array(errors)
        good = errors[errors < 100]
        print(f'{gname:<12} {pname:<10} | {successes:>3}/{N_test:<3} | '
              f'{good.mean() if len(good) else 999:>10.6f} | {good.max() if len(good) else 999:>10.6f} | '
              f'{np.mean(times)*1000:>8.2f} ms')

## Prototype v2: Use Qiskit XXDecomposer directly on V = CAN(T) · C⁻¹

The previous prototype failed because manual KAK-peeling doesn't handle 
Weyl reflections/shifts. The XXDecomposer does this correctly via 
`decompose_xxyy_into_xxyy_xx` (arccos formula + phase solver).

Pipeline:
1. Form V = CAN(T) · C⁻¹  
2. Feed V to XXDecomposer with sub-gate strengths as the basis  
3. XXDecomposer returns a fully analytical circuit  
4. Prepend C, verify we reach T

In [ ]:
from qiskit.synthesis import XXDecomposer
from qiskit.quantum_info import Operator
import time

def solve_segment_analytical(C_matrix, G_weyl, T_weyl):
    """
    Solve L(C, G, T) analytically using XXDecomposer on V = CAN(T) · C^{-1}.
    
    Returns: (fidelity_error, circuit, wall_time_ms)
    """
    c1, c2, c3 = G_weyl
    
    # Sub-gate strengths: XX-type interaction strength = pi/2 * weyl_c1
    # For XX(c): strength = pi/2 * c  (in the XXDecomposer convention)
    # YY(c) and ZZ(c) are locally equiv to XX(c), same strength
    strengths = []
    if abs(c3) > 1e-10:
        strengths.append(np.pi / 2 * c3)
    if abs(c2) > 1e-10:
        strengths.append(np.pi / 2 * c2)
    if abs(c1) > 1e-10:
        strengths.append(np.pi / 2 * c1)
    
    # Build XXDecomposer with these strengths
    basis_fidelity = {s: 1.0 for s in strengths}
    decomposer = XXDecomposer(basis_fidelity=basis_fidelity)
    
    # V = CAN(T) · C^{-1}
    CAN_T = canonical_matrix(*T_weyl)
    C_inv = np.linalg.inv(C_matrix)
    V = CAN_T @ C_inv
    
    t0 = time.perf_counter()
    circuit = decomposer(Operator(V))
    dt = (time.perf_counter() - t0) * 1000
    
    # Verify: circuit should implement V
    V_reconstructed = Operator(circuit).data
    
    # Process fidelity
    fid = abs(np.trace(V_reconstructed.conj().T @ V)) / 4
    
    return 1 - fid, circuit, dt

# --- Quick test ---
rng = np.random.default_rng(42)
C = canonical_matrix(0.3, 0.2, 0.1)
u = random_local(rng)
G = canonical_matrix(0.5, 0.5, 0.0)  # iSWAP
T_matrix = G @ u @ C
T_weyl = tuple(canonical_weyl(T_matrix))

print("=== Single test: iSWAP segment ===")
print(f"C = {canonical_weyl(C)}, T = {list(T_weyl)}")

err, circ, dt = solve_segment_analytical(C, (0.5, 0.5, 0.0), T_weyl)
print(f"Infidelity: {err:.2e}")
print(f"Time: {dt:.2f} ms")
print(f"Circuit depth: {circ.depth()}, 2Q gates: {circ.num_nonlocal_gates()}")
print(circ)

In [ ]:
# --- Mass test ---
N_test = 200

gate_configs = [
    ('iSWAP',     (0.5, 0.5, 0.0)),
    ('sq2iswap',  (0.25, 0.25, 0.0)),
    ('B gate',    (0.5, 0.25, 0.0)),
    ('CX',        (0.5, 0.0, 0.0)),
    ('generic',   (0.4, 0.3, 0.15)),
]

prefix_configs = [
    ('generic', canonical_matrix(0.3, 0.2, 0.1)),
    ('C=I',     np.eye(4, dtype=complex)),
]

print(f"{'Gate':<12} {'Prefix':<10} | {'Success':>7} | {'Mean infid':>12} | {'Max infid':>12} | {'Mean 2Q':>7} | {'Time ms':>8}")
print('='*85)

for gname, G_weyl in gate_configs:
    for pname, C in prefix_configs:
        rng = np.random.default_rng(0)
        infids = []
        n2q = []
        times = []
        success = 0
        
        for i in range(N_test):
            u = random_local(rng)
            G = canonical_matrix(*G_weyl)
            T_matrix = G @ u @ C
            T_w = tuple(canonical_weyl(T_matrix))
            
            try:
                err, circ, dt = solve_segment_analytical(C, G_weyl, T_w)
                infids.append(err)
                n2q.append(circ.num_nonlocal_gates())
                times.append(dt)
                if err < 1e-10:
                    success += 1
            except Exception as e:
                infids.append(999)
                times.append(0)
                n2q.append(-1)
        
        infids = np.array(infids)
        good = infids[infids < 100]
        n2q_good = [x for x in n2q if x >= 0]
        
        print(f'{gname:<12} {pname:<10} | {success:>3}/{N_test:<3} | '
              f'{np.mean(good) if len(good) else -1:>12.2e} | '
              f'{np.max(good) if len(good) else -1:>12.2e} | '
              f'{np.mean(n2q_good) if n2q_good else -1:>7.1f} | '
              f'{np.mean(times):>7.2f}')

In [ ]:
# === Commutative square: measure what the analytical XX solve actually gives us ===
#
# Core question: if we solve L(C, XX(a), M1) analytically (via xx_circuit_step),
# we get M1_actual as a specific 4x4 matrix. Then T_check = YY(b) · M1_actual.
# Does Weyl(T_check) match the original T_weyl?
#
# This is the test of whether the analytical XX solve + direct YY propagation 
# equals the original segment solve.

from qiskit.synthesis.two_qubit.xx_decompose.circuits import xx_circuit_step
from qiskit.synthesis.two_qubit.two_qubit_decompose import TwoQubitWeylDecomposition
from qiskit.quantum_info import Operator
from qiskit.circuit import QuantumCircuit as QC

def measure_propagation(C_matrix, G_weyl, n_samples=200, seed=0):
    """
    For random segments L(C, G, T):
    1. Compute the TRUE M1 = XX·u·C (we know u since we generated the problem)
    2. Solve L(C, XX, M1_weyl) analytically via xx_circuit_step
    3. Get M1_analytical (the actual matrix from the analytical solve)
    4. Propagate: T_check = YY · M1_analytical
    5. Compare Weyl(T_check) vs T_weyl
    
    This measures: does the analytical XX solve give the RIGHT M1_actual
    such that direct YY propagation produces T?
    """
    a, b, c3 = G_weyl
    XX = canonical_matrix(a, 0.0, 0.0)
    YY = canonical_matrix(0.0, b, 0.0)
    
    strength = a * np.pi / 2
    embodiment = QC(2)
    embodiment.rzx(2 * strength, 0, 1)
    
    rng = np.random.default_rng(seed)
    results = []
    
    for i in range(n_samples):
        u = random_local(rng)
        
        # Ground truth
        T_actual = XX @ YY @ u @ C_matrix  # = G·u·C
        T_weyl = canonical_weyl(T_actual)
        M1_true = XX @ u @ C_matrix  # true intermediate
        M1_weyl = canonical_weyl(M1_true)
        
        # Step 1: Solve L(C, XX, M1_weyl) analytically
        c_weyl = canonical_weyl(C_matrix)
        source = np.array(c_weyl) * np.pi / 2
        target = np.array(M1_weyl) * np.pi / 2
        
        try:
            step = xx_circuit_step(source, strength, target, embodiment)
        except Exception:
            results.append({'success': False, 'err': float('inf'), 'reason': 'step_fail'})
            continue
        
        # Step 2: Get M1_analytical as actual matrix
        prefix_mat = Operator(step['prefix_circuit']).data
        affix_mat = Operator(step['affix_circuit']).data
        
        # The step decomposes: CAN(M1) = affix · CAN(C_weyl) · prefix
        # But C is not CAN(C_weyl), it's C = K1 · CAN(C_weyl) · K2
        # So we need: M1_analytical = affix · K1^{-1} · C · K2^{-1} · prefix
        C_kak = TwoQubitWeylDecomposition(Operator(C_matrix))
        K1 = np.kron(C_kak.K1r, C_kak.K1l)
        K2 = np.kron(C_kak.K2r, C_kak.K2l)
        
        M1_analytical = affix_mat @ np.linalg.inv(K1) @ C_matrix @ np.linalg.inv(K2) @ prefix_mat
        
        # Check M1 Weyl matches
        M1_ana_weyl = canonical_weyl(M1_analytical)
        m1_err = np.max(np.abs(M1_ana_weyl - M1_weyl))
        
        # Step 3: Propagate through YY
        T_check = YY @ M1_analytical
        T_check_weyl = canonical_weyl(T_check)
        t_err = np.max(np.abs(T_check_weyl - T_weyl))
        
        # Also check Frobenius distance to true T
        # T_actual and T_check should be locally equivalent (same Weyl)
        # but might differ in local frame
        
        results.append({
            'success': t_err < 1e-6,
            'err': t_err,
            'm1_err': m1_err,
            'reason': 'ok',
        })
    
    return results

# --- Run for iSWAP ---
C = canonical_matrix(0.3, 0.2, 0.1)

configs = [
    ('iSWAP',    (0.5, 0.5, 0.0)),
    ('sq2iswap', (0.25, 0.25, 0.0)),
    ('B gate',   (0.5, 0.25, 0.0)),
]

print(f"{'Gate':<12} | {'XX step ok':>10} | {'T propagation':>13} | {'mean T err':>10} | {'max T err':>10} | {'mean M1 err':>11}")
print('='*80)

for gname, gw in configs:
    results = measure_propagation(C, gw, n_samples=500, seed=42)
    
    step_ok = sum(1 for r in results if r['reason'] == 'ok')
    t_success = sum(1 for r in results if r['success'])
    errs = [r['err'] for r in results if r['reason'] == 'ok']
    m1_errs = [r['m1_err'] for r in results if r['reason'] == 'ok']
    
    print(f'{gname:<12} | {step_ok:>5}/{500:<4} | {t_success:>8}/{step_ok:<4} | '
          f'{np.mean(errs):>10.2e} | {np.max(errs):>10.2e} | {np.mean(m1_errs):>11.2e}')